在调用模型时（如使用 `invoke()`, `ainvoke()`, `stream()`, `batch()` 等方法时），我们可以传入`config`参数。

```python
def invoke(
    self,
    input: LanguageModelInput,
    config: RunnableConfig | None = None,
    *,
    stop: list[str] | None = None,
    **kwargs: Any,
) -> AIMessage
```

`config`参数：允许在调用模型时，动态地配置和控制模型的行为，而无需在初始化时就固定所有参数，这为应用带来了极大的灵活性和可维护性。

关于`config`中可配参数的解释参考：
https://reference.langchain.com/python/langchain-core/runnables/config/RunnableConfig

**config中支持配置的参数如下：**

| 配置项 | 类型 | 描述 |
| ---- | ---- | ---- |
| run_name | str | 为当前运行设置一个可读的名称。如在LangSmith追踪系统中快速定位和识别不同的运行任务。 |
| tags | List[str] | 为运行设置标签，用于分类和过滤。如在LangSmith追踪系统中快速定位和识别不同的运行任务。 |
| callbacks | List[BaseCallbackHandler] | 设置回调处理器，在运行的不同阶段（开始、流输出、结束等）触发。与一些监控平台（如LangSmith）集成进行深度追踪和调试。 |
| metadata | Dict[str,Any] | 附加任意的键值对元数据。记录本次调用的业务上下文，如`{"user_id": "123","session_id": "abc"}` |
| max_concurrency | int | 限制当前可运行对象的最大并发运行数。防止对API接口或本地资源造成过大压力，实现简单的速率限制。 |
| recursion_limit | int | 限制运行时递归调用的最大深度。主要在复杂的工作流（如Agent执行多步工具调用）中，防止出现无限递归循环。 |
| configurable | Dict[Str,Any] | 一个万能字典，用于传递其他可配置参数。实现更高级的动态行为，如配置可替代的模型或组件。 |

**说明如下：**
- config中参数 `run_name`、`tags`、`callbacks` 主要用在LangSmith中，用于追踪、筛选和调试。
- `metadata` 可以配置用户指定的一些信息，在工作流开发中，当整个流程被包装为Runnable链时，可以将这些参数传递给后续的链节点使用。
- `configurable` 中可配置的参数与 `init_chat_model` 初始化模型参数一样，与在初始化模型时设置的参数（如 `temperature=0.7`）的关键区别在于：
  - `init_chat_model`初始化参数：模型的**默认设置**，适用于该模型实例的大部分场景。
  - 运行时 config：**单次调用的特定设置**，优先级更高，针对本次调用进行特殊调整。

配置configurable覆盖默认参数时需要在“init_chat_model”初始化模型中指定
“configurable_fields”参数来指定模型运行时可替换的参数有哪些。



In [3]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv(override=True)


# 1. 初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    temperature=0.2,
    max_tokens=500,
    # 指定可调整参数
    configurable_fields=("model", "model_provider", "temperature", "max_tokens"),
)

# 2. 准备 config 字典
config = {
    "run_name": "joke_generation",
    "tags": ["tag1", "tag2"],
    "metadata": {"user_id": "123"},
    "configurable": {
        "model": "deepseek-v4-pro",
        "model_provider": "deepseek",
        "temperature": 0.7,
        "max_tokens": 1000,
    },
}

# 3. 调用模型并传入config
response = model.invoke("1 + 2 = ？", config=config)
rprint(response)

AIMessage(
    content='1 + 2 = 3',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': '我们被问到 "1 + 2 = ？" 这是一个简单的算术问题。答案是 3。用中文回答即可。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 35,
            'prompt_tokens': 11,
            'total_tokens': 46,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 27,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 11
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-pro',
        'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402',
        'id': '69c6903b-9027-44d0-9067-0a09b93f628c',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f186e-6263-7601-bb41-baa322e39e48-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 11,
        'output_tokens': 35,
        'total_tokens': 46,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 27}
    }
)

## `configurable.thread_id` 与 Checkpointer

除了动态覆盖模型参数，`config` 中的 `configurable` 也常用于给 LangGraph / Agent 运行时传递会话级配置。

最典型的例子是：

```python
config={"configurable": {"thread_id": "session-1"}}
```

`thread_id` 不是模型生成参数，不会直接传给 LLM 控制温度、token 数等行为。它是运行时用来标识一条会话 / 线程的 ID。

当 Agent 或 LangGraph graph 配置了 checkpointer 后，运行状态会按 `thread_id` 保存和恢复。相同 `thread_id` 的后续调用可以读取前面轮次的消息历史；不同 `thread_id` 则表示不同会话。

本地学习和单元测试时可以使用 `InMemorySaver`。它只保存在当前 Python 进程内，进程结束后数据会丢失；生产环境通常应换成 SQLite、Postgres 等持久化 checkpointer。


In [ ]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from rich import print as rprint

load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter_base_url = os.getenv("OPENROUTER_BASE_URL")

model = init_chat_model(
    api_key=openrouter_api_key,
    base_url=openrouter_base_url,
    model="gpt-5.4-mini",
)

# checkpointer 负责保存 Agent/Graph 的运行状态。
# InMemorySaver 适合本地学习和测试；它只在当前 Python 进程内有效。
agent = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver(),
)

# thread_id 用来标识一条会话。
# 相同 thread_id 的多次调用会复用同一条 checkpoint / 消息历史。
config = {"configurable": {"thread_id": "session-1"}}

first_result = agent.invoke(
    {"messages": [{"role": "user", "content": "我叫张三，请记住我的名字。"}]},
    config=config,
)

# 第二次调用复用相同 thread_id，因此 Agent 可以从 checkpointer 中恢复前一轮消息历史。
second_result = agent.invoke(
    {"messages": [{"role": "user", "content": "我叫什么名字？"}]},
    config=config,
)

rprint(second_result)
